# Arabic Restaurant Complaints Classifier — Colab demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FerasMad/NLP-complaints-system/blob/main/notebooks/colab_demo.ipynb)

Run the model in Google Colab without a local GPU. Free tier works.

**Before you run:** switch to a GPU runtime — *Runtime → Change runtime type → T4 GPU*. CPU works too, just ~4x slower per request.

Source repo: https://github.com/FerasMad/NLP-complaints-system

## 1. Install

In [ ]:
!pip install -q transformers torch gradio

## 2. Load the model from HuggingFace Hub

Pulls the single best CAMeLBERT-mix model (~440 MB, 94.86% test accuracy). For the full 4-model ensemble (~1.8 GB, 95.05% accuracy), see the repo README.

Replace `HF_REPO` below if you uploaded the model under a different name.

In [ ]:
import re, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

HF_REPO = "FerasMad/arabic-complaints-classifier"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading on {device}...")

tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
model = AutoModelForSequenceClassification.from_pretrained(HF_REPO).to(device).eval()

CATEGORIES = [
    "التوصيل", "السعر والقيمة", "النظافة", "جودة الطعام",
    "خدمة الموظفين", "دقة الطلب", "عامة", "وقت الانتظار",
]

print("Ready.")

## 3. Predict

The cleaning function below matches the one used in training (tashkeel removal, alef/ya/ta-marbuta normalization). Skipping it costs ~1–2% accuracy.

In [ ]:
TASHKEEL = re.compile(r"[\u064B-\u065F]")
NON_ARABIC = re.compile(r"[^\u0600-\u06FFa-zA-Z0-9\u0660-\u0669\s]")
WHITESPACE = re.compile(r"\s+")

def clean(text):
    t = TASHKEEL.sub("", text)
    t = t.translate(str.maketrans({"أ":"ا","إ":"ا","آ":"ا","ٱ":"ا","ى":"ي","ة":"ه"}))
    t = NON_ARABIC.sub(" ", t)
    return WHITESPACE.sub(" ", t).strip().lower()

@torch.no_grad()
def predict(text, top_k=3):
    enc = tokenizer(clean(text), return_tensors="pt", truncation=True, max_length=192).to(device)
    probs = torch.softmax(model(**enc).logits[0], dim=-1).cpu().numpy()
    top = probs.argsort()[::-1][:top_k]
    return [(CATEGORIES[i], float(probs[i])) for i in top]

examples = [
    "الاكل بايخ ومالح",
    "وصل الطلب بارد والمندوب تاخر ساعتين",
    "الموظف اسلوبه سيء وغير محترم",
    "النظافه سيئه الطاولات متسخه",
    "الاسعار مبالغ فيها لا تناسب الجوده",
    "طلبت برجر بدون بصل لكنهم وضعوه",
]

for text in examples:
    print(f"\n{text}")
    for cat, score in predict(text):
        print(f"  {cat:<15} {score:.2%}")

## 4. Public Gradio demo (optional)

Launches a temporary public `*.gradio.live` URL. Lasts 72 hours. No account needed. Share the link with anyone.

In [ ]:
import gradio as gr

EXAMPLES = [
    "وصل الطلب بارد جدا والمندوب تاخر اكثر من ساعتين",
    "الاسعار مبالغ فيها لا تناسب الجوده",
    "النظافه سيئه الطاولات متسخه",
    "طلبت برجر بدون بصل لكنهم وضعوه",
    "انتظرت ساعه كامله قبل ان ياتي طلبي",
    "الموظف اسلوبه سيء وغير محترم",
    "الاكل بايخ ومالح",
    "تجربه سيئه عموما لن اعود",
]

def gradio_predict(text):
    if not text or not text.strip():
        return {}
    return {cat: score for cat, score in predict(text)}

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=4, label="اكتب الشكوى", placeholder="مثال: الاكل بايخ ومالح", rtl=True),
    outputs=gr.Label(num_top_classes=3, label="التصنيف"),
    examples=EXAMPLES,
    title="تصنيف شكاوى المطاعم العربية",
    description="4-model BERT ensemble · 8 categories · Saudi-Gulf dialect · 95% test accuracy",
)

demo.launch(share=True)

## Categories

| ID | Arabic | English |
|----|--------|---------|
| 0 | التوصيل | Delivery |
| 1 | السعر والقيمة | Price / value |
| 2 | النظافة | Cleanliness |
| 3 | جودة الطعام | Food quality |
| 4 | خدمة الموظفين | Staff service |
| 5 | دقة الطلب | Order accuracy |
| 6 | عامة | General |
| 7 | وقت الانتظار | Wait time |

Source repo: https://github.com/FerasMad/NLP-complaints-system